# Load Data

In [37]:
import os

# path to features directory -- update if necessary
DATA_DIR = os.path.join('..', '..', 'data', 'features')

In [38]:
import pandas as pd

train_path = os.path.join(DATA_DIR, 'train.csv')
train_df = pd.read_csv(train_path)

val_path = os.path.join(DATA_DIR, 'val.csv')
val_df = pd.read_csv(val_path)

test_path = os.path.join(DATA_DIR, 'test.csv')
test_df = pd.read_csv(test_path)

train_df.head()

,file_name,zcrs_0,zcrs_1,zcrs_2,zcrs_3,zcrs_4,zcrs_5,zcrs_6,zcrs_7,zcrs_8,...,mfcc_max_10,mfcc_mean_11,mfcc_std_11,mfcc_min_11,mfcc_max_11,mfcc_mean_12,mfcc_std_12,mfcc_min_12,mfcc_max_12,label
0,1079_ITH_ANG_XX.mp3,0.078317,0.061731,0.087380,0.045828,0.080027,0.076094,0.044631,0.055917,0.084473,...,20.121860,2.642711,5.123800,-7.284746,16.696190,0.078399,5.888541,-9.171087,18.859655,ANG
1,1056_IWW_ANG_XX.mp3,0.087303,0.054106,0.072861,0.084070,0.063591,0.046346,0.241431,0.059280,0.065962,...,4.777219,-0.960902,8.551895,-21.385560,22.197117,1.897318,7.625268,-16.679660,15.909523,ANG
2,1043_IWW_ANG_XX.mp3,0.111986,0.059562,0.069161,0.053409,0.097711,0.044056,0.086143,0.052670,0.149397,...,25.344017,-2.454590,9.791595,-20.133453,26.196903,-5.470756,5.950812,-20.356094,13.582540,ANG
3,1061_MTI_ANG_XX.mp3,0.068037,0.071055,0.084756,0.084175,0.075700,0.046209,0.085917,0.064786,0.111808,...,20.021273,1.821852,11.888531,-22.822205,32.116337,0.774488,9.480104,-30.069153,33.686405,ANG
4,1074_MTI_ANG_XX.mp3,0.074074,0.040523,0.083370,0.131736,0.076834,0.061147,0.047204,0.060131,0.045606,...,25.050407,-2.314029,9.899802,-27.713839,24.185566,-4.683256,11.325140,-38.701878,19.316269,ANG


# Preprocessing

In [39]:
# number of nan values in each column
print(train_df.isna().sum().sort_values(ascending=False))

# drop columns with nan (just 4 columns)
train_df = train_df.dropna(axis=1)

train_df = train_df.drop(columns=['file_name'])

centroid_skew     25
centroid_kurt     25
bandwidth_skew    25
bandwidth_kurt    25
mfcc_min_5         0
                  ..
bandwidth_min      0
bandwidth_std      0
bandwidth_mean     0
centroid_max       0
label              0
Length: 90, dtype: int64


In [5]:
X_train = train_df.drop(columns=['label'])
y_train = train_df['label']

features = X_train.columns

X_val = val_df[features]
y_val = val_df['label']

In [6]:
classes = ['ANG', 'DIS', 'FEA', 'HAP', 'NEU', 'SAD']
class_to_num = {cls: i for i, cls in enumerate(classes)}
num_to_class = {i: cls for i, cls in enumerate(classes)}

y_train_num = [class_to_num[cls] for cls in y_train]
y_val_num = [class_to_num[cls] for cls in y_val]

# Logistic Regression

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import itertools
from tqdm import tqdm

Cs = [0.01, 0.05, 0.1]
penalties = ['l1', 'l2'] 
solvers = ['liblinear', 'saga']
max_iters = [50000]

best_score = 0
best_params = {}

combos = itertools.product(Cs, penalties, solvers, max_iters)

for C, penalty, solver, max_iter in tqdm(combos):
    if penalty == 'l2' and solver == 'liblinear':
        continue

    lr = LogisticRegression(C=C, penalty=penalty, solver=solver, max_iter=max_iter)
    lr.fit(X_train, y_train)
    score = accuracy_score(y_val, lr.predict(X_val))
    
    print(f"C: {C}, penalty: {penalty}, solver: {solver}, max_iter: {max_iter}, score: {score}")
    if score > best_score:
        best_score = score
        best_params = {'C': C, 'penalty': penalty, 'solver': solver, 'max_iter': max_iter}

print(f"Best validation accuracy: {best_score:.4f}")
print(f"Best parameters: {best_params}")

lr = LogisticRegression(max_iter=best_params['max_iter'],
                        C=best_params['C'],
                        penalty=best_params['penalty'],
                        solver=best_params['solver'])
lr.fit(X_train, y_train)

1it [00:23, 23.35s/it]

C: 0.01, penalty: l1, solver: liblinear, max_iter: 50000, score: 0.478494623655914


2it [01:20, 43.40s/it]

C: 0.01, penalty: l1, solver: saga, max_iter: 50000, score: 0.4946236559139785


4it [01:56, 27.37s/it]

...

10it [05:45, 38.77s/it]

C: 0.1, penalty: l1, solver: saga, max_iter: 50000, score: 0.48655913978494625


12it [06:21, 31.80s/it]

C: 0.1, penalty: l2, solver: saga, max_iter: 50000, score: 0.48655913978494625
Best validation accuracy: 0.4946
Best parameters: {'C': 0.01, 'penalty': 'l1', 'solver': 'saga', 'max_iter': 50000}


LogisticRegression(C=0.01, max_iter=50000, penalty='l1', solver='saga')

# Random Forest

In [14]:
from sklearn.ensemble import RandomForestClassifier

n_estimators = [500, 1000]
max_depth = [None, 20]
min_samples_split = [2, 5]
min_samples_leaf = [1, 2, 4]

best_score = 0
best_params = {}

combos = itertools.product(n_estimators, max_depth, min_samples_split, min_samples_leaf)

for n_estimators, max_depth, min_samples_split, min_samples_leaf in tqdm(combos):
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth, 
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    rf.fit(X_train, y_train)
    score = accuracy_score(y_val, rf.predict(X_val))
    print(f"n_estimators: {n_estimators}, max_depth: {max_depth}, min_samples_split: {min_samples_split}, min_samples_leaf: {min_samples_leaf}, score: {score}")
    if score > best_score:
        best_score = score
        best_params = {
            'n_estimators': n_estimators,
            'max_depth': max_depth,
            'min_samples_split': min_samples_split,
            'min_samples_leaf': min_samples_leaf
        }

print(f"Best validation accuracy: {best_score:.4f}")
rf = RandomForestClassifier(n_estimators=best_params['n_estimators'],
                            max_depth=best_params['max_depth'],
                            min_samples_split=best_params['min_samples_split'],
                            min_samples_leaf=best_params['min_samples_leaf'],
                            random_state=42)
rf.fit(X_train, y_train)

1it [00:14, 14.61s/it]

n_estimators: 500, max_depth: None, min_samples_split: 2, min_samples_leaf: 1, score: 0.5228494623655914


2it [00:27, 13.49s/it]

n_estimators: 500, max_depth: None, min_samples_split: 2, min_samples_leaf: 2, score: 0.5309139784946236


3it [00:39, 12.73s/it]

...

23it [07:36, 25.51s/it]

n_estimators: 1000, max_depth: 20, min_samples_split: 5, min_samples_leaf: 2, score: 0.5188172043010753


24it [08:00, 20.02s/it]

n_estimators: 1000, max_depth: 20, min_samples_split: 5, min_samples_leaf: 4, score: 0.521505376344086
Best validation accuracy: 0.5309


RandomForestClassifier(min_samples_leaf=2, n_estimators=500, random_state=42)

# Perceptron

In [21]:
from sklearn.neural_network import MLPClassifier

learning_rates = ['constant', 'adaptive']
alphas = [1e-4, 1e-3]
max_iters = [300, 500]
n_iter_no_change = [5, 10, 20]

best_score = 0
best_params = {}

combos = itertools.product(learning_rates, alphas, max_iters, n_iter_no_change)

for learning_rate, alpha, max_iter, n_iter_no_change in tqdm(combos):
    mlp = MLPClassifier(
        hidden_layer_sizes=(128, 128, 128, 128),
        activation='relu',
        solver='adam',
        alpha=alpha,
        learning_rate=learning_rate,
        max_iter=max_iter,
        random_state=42,
        n_iter_no_change=n_iter_no_change,
        verbose=False
    )

    mlp.fit(X_train, y_train)

    score = accuracy_score(y_val, mlp.predict(X_val))
    print(f"learning_rate: {learning_rate}, alpha: {alpha}, max_iter: {max_iter}, n_iter_no_change: {n_iter_no_change}, score: {score}")
    
    if score > best_score:
        best_score = score
        best_params = {
            'learning_rate': learning_rate,
            'alpha': alpha,
            'max_iter': max_iter,
            'n_iter_no_change': n_iter_no_change
        }

print(f"Best validation accuracy: {best_score:.4f}")

1it [00:04,  4.32s/it]

learning_rate: constant, alpha: 0.0001, max_iter: 300, n_iter_no_change: 5, score: 0.3602150537634409


2it [00:12,  6.68s/it]

learning_rate: constant, alpha: 0.0001, max_iter: 300, n_iter_no_change: 10, score: 0.4206989247311828


/Users/reyanshbahl/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
3it [00:45, 18.70s/it]

...

learning_rate: adaptive, alpha: 0.001, max_iter: 500, n_iter_no_change: 5, score: 0.3629032258064516


23it [05:08,  8.64s/it]

learning_rate: adaptive, alpha: 0.001, max_iter: 500, n_iter_no_change: 10, score: 0.43548387096774194


24it [05:24, 13.53s/it]

learning_rate: adaptive, alpha: 0.001, max_iter: 500, n_iter_no_change: 20, score: 0.4583333333333333
Best validation accuracy: 0.4772


In [22]:
mlp = MLPClassifier(
        hidden_layer_sizes=(128, 128, 128, 128),
        activation='relu',
        solver='adam',
        alpha=best_params['alpha'],
        learning_rate=best_params['learning_rate'],
        max_iter=best_params['max_iter'],
        random_state=42,
        n_iter_no_change=best_params['n_iter_no_change'],
        verbose=False
    )
mlp.fit(X_train, y_train)

/Users/reyanshbahl/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier(hidden_layer_sizes=(128, 128, 128, 128), max_iter=300,
              n_iter_no_change=20, random_state=42)

# XGBoost

In [26]:
from xgboost import XGBClassifier

n_estimators = [500, 1000]
max_depth = [12, 18]
learning_rate = [0.001, 0.005]

best_score = 0
best_params = {}

combos = itertools.product(n_estimators, max_depth, learning_rate)

for n_estimators, max_depth, learning_rate in tqdm(combos):
    xgb = XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        random_state=42,
        early_stopping_rounds=50
    )

    xgb.fit(X_train, y_train_num, 
            eval_set=[(X_val, y_val_num)],
            verbose=False)

    score = accuracy_score(y_val_num, xgb.predict(X_val))
    print(f"n_estimators: {n_estimators}, max_depth: {max_depth}, learning_rate: {learning_rate}, score: {score}")
    if score > best_score:
        best_score = score
        best_params = {
            'n_estimators': n_estimators,
            'max_depth': max_depth,
            'learning_rate': learning_rate
        }

print(f"Best validation accuracy: {best_score:.4f}")

1it [00:41, 41.67s/it]

n_estimators: 500, max_depth: 12, learning_rate: 0.001, score: 0.4650537634408602


2it [01:20, 40.04s/it]

n_estimators: 500, max_depth: 12, learning_rate: 0.005, score: 0.5053763440860215


3it [02:14, 46.16s/it]

...

7it [07:23, 77.60s/it]

n_estimators: 1000, max_depth: 18, learning_rate: 0.001, score: 0.4798387096774194


8it [08:48, 66.02s/it]

n_estimators: 1000, max_depth: 18, learning_rate: 0.005, score: 0.5255376344086021
Best validation accuracy: 0.5309


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.005, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=12, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [28]:
xgb = XGBClassifier(
    n_estimators=5000,
    max_depth=best_params['max_depth'],
    learning_rate=best_params['learning_rate'])

xgb.fit(X_train, y_train_num, 
            eval_set=[(X_val, y_val_num)],
            verbose=50)

[0]	validation_0-mlogloss:1.78855
[50]	validation_0-mlogloss:1.65732
[100]	validation_0-mlogloss:1.56655
[150]	validation_0-mlogloss:1.49887
[200]	validation_0-mlogloss:1.44610
[250]	validation_0-mlogloss:1.40583
[300]	validation_0-mlogloss:1.37221
[350]	validation_0-mlogloss:1.34532
[400]	validation_0-mlogloss:1.32546
[450]	validation_0-mlogloss:1.30608
[500]	validation_0-mlogloss:1.28876
[550]	validation_0-mlogloss:1.27401
[600]	validation_0-mlogloss:1.26037
[650]	validation_0-mlogloss:1.25043
[700]	validation_0-mlogloss:1.24337
[750]	validation_0-mlogloss:1.23562
[800]	validation_0-mlogloss:1.22802
[850]	validation_0-mlogloss:1.22073
[900]	validation_0-mlogloss:1.21603
[950]	validation_0-mlogloss:1.21322
[1000]	validation_0-mlogloss:1.20948
[1050]	validation_0-mlogloss:1.20713
[1100]	validation_0-mlogloss:1.20469
[1150]	validation_0-mlogloss:1.20279
[1200]	validation_0-mlogloss:1.20186
[1250]	validation_0-mlogloss:1.20145
[1300]	validation_0-mlogloss:1.20056
[1350]	validation_0-mlog

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.005, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=12, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=5000, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

# Light GBM

In [ ]:
import lightgbm as lgb

lgb_train = lgb.Dataset(X_train, y_train_num)
lgb_eval = lgb.Dataset(X_val, y_val_num, reference=lgb_train)

params = {
    'objective': 'multiclass',
    'num_class': 6,
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.005,
    'max_depth': 18,
    'random_state': 42,
    'early_stopping_rounds': 50,
    'num_leaves': 127,
    'verbose': 0
}

gbm = lgb.train(params,
               lgb_train,
               valid_sets=lgb_eval,
               num_boost_round=5000,
               callbacks=[lgb.early_stopping(stopping_rounds=50)])

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds


# Save Models

In [34]:
import pickle

MODELS_DIR = os.path.join('..', '..', 'models')

os.makedirs(MODELS_DIR, exist_ok=True)

with open(os.path.join(MODELS_DIR, 'lr.pkl'), 'wb') as f:
    pickle.dump(lr, f)

with open(os.path.join(MODELS_DIR, 'rf.pkl'), 'wb') as f:
    pickle.dump(rf, f)

with open(os.path.join(MODELS_DIR, 'mlp.pkl'), 'wb') as f:
    pickle.dump(mlp, f)

with open(os.path.join(MODELS_DIR, 'xgb.pkl'), 'wb') as f:
    pickle.dump(xgb, f)

with open(os.path.join(MODELS_DIR, 'lgb.pkl'), 'wb') as f:
    pickle.dump(gbm, f)